In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, round as spark_round

spark = SparkSession.builder.appName("Tugas11-KopiNusantara").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 17:12:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)



+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



In [2]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

idx_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

model_idx_menu = idx_menu.fit(df)
model_idx_cabang = idx_cabang.fit(df)
model_idx_waktu = idx_waktu.fit(df)

df_indexed = model_idx_menu.transform(df)
df_indexed = model_idx_cabang.transform(df_indexed)
df_indexed = model_idx_waktu.transform(df_indexed)

fitur_lengkap = ["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"]
assembler = VectorAssembler(inputCols=fitur_lengkap, outputCol="features")
df_final = assembler.transform(df_indexed)

df_final.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan",
                "nama_menu_idx", "cabang_idx", "waktu_pesan_idx", "features").show(5, truncate=False)

+--------------+----------+-----------+---+------------+-------------+----------+---------------+-------------------------+
|nama_menu     |cabang    |waktu_pesan|qty|harga_satuan|nama_menu_idx|cabang_idx|waktu_pesan_idx|features                 |
+--------------+----------+-----------+---+------------+-------------+----------+---------------+-------------------------+
|Kentang Goreng|Semarang  |Malam      |1  |30000       |5.0          |2.0       |1.0            |[1.0,30000.0,5.0,2.0,1.0]|
|Kentang Goreng|Yogyakarta|Sore       |3  |20000       |5.0          |1.0       |3.0            |[3.0,20000.0,5.0,1.0,3.0]|
|Kentang Goreng|Semarang  |Siang      |5  |22000       |5.0          |2.0       |2.0            |[5.0,22000.0,5.0,2.0,2.0]|
|Kentang Goreng|Yogyakarta|Pagi       |2  |20000       |5.0          |1.0       |0.0            |[2.0,20000.0,5.0,1.0,0.0]|
|Roti Bakar    |Magelang  |Malam      |2  |30000       |4.0          |0.0       |1.0            |[2.0,30000.0,4.0,0.0,1.0]|
+-------

In [3]:
from pyspark.ml.regression import LinearRegression

train, test = df_final.randomSplit([0.75, 0.25], seed=101)
print(f"Training: {train.count()} | Testing: {test.count()}")

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)

print("Urutan fitur :", fitur_lengkap)
print("Koefisien    :", model.coefficients)
print("Intercept    :", model.intercept)

Training: 1839 | Testing: 661
Urutan fitur : ['qty', 'harga_satuan', 'nama_menu_idx', 'cabang_idx', 'waktu_pesan_idx']
Koefisien    : [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept    : -66903.17766485889


In [4]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)

ev_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
ev_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse_utama = ev_rmse.evaluate(predictions)
r2_utama = ev_r2.evaluate(predictions)
rata2 = test.agg(avg("total_penjualan")).first()[0]

print(f"RMSE : Rp {rmse_utama:,.0f}")
print(f"R²   : {r2_utama:.4f}")
print(f"RMSE relatif terhadap rata-rata pesanan: {rmse_utama / rata2 * 100:.1f}%")

RMSE : Rp 10,901
R²   : 0.9107
RMSE relatif terhadap rata-rata pesanan: 16.7%


In [5]:
assembler_dua = VectorAssembler(inputCols=["qty", "harga_satuan"], outputCol="features_dua")
train_dua = assembler_dua.transform(train)
test_dua = assembler_dua.transform(test)

model_dua = LinearRegression(featuresCol="features_dua", labelCol="total_penjualan").fit(train_dua)
pred_dua = model_dua.transform(test_dua)

rmse_dua = ev_rmse.evaluate(pred_dua)
r2_dua = ev_r2.evaluate(pred_dua)

print(f"{'Model':<40}{'R²':>9}{'RMSE (Rp)':>14}")
print(f"{'Utama (5 fitur)':<40}{r2_utama:>9.4f}{rmse_utama:>14,.0f}")
print(f"{'Kedua (qty, harga saja)':<40}{r2_dua:>9.4f}{rmse_dua:>14,.0f}")
print(f"\nSelisih R² (kedua - utama): {r2_dua - r2_utama:+.4f}")

Model                                          R²     RMSE (Rp)
Utama (5 fitur)                            0.9107        10,901
Kedua (qty, harga saja)                    0.9108        10,896

Selisih R² (kedua - utama): +0.0001


In [6]:
skenario = [
    ("Kopi Susu",  "Magelang",   "Pagi",  2, 18000),
    ("Roti Bakar", "Yogyakarta", "Siang", 3, 25000),
    ("Es Teh",     "Semarang",   "Malam", 5, 15000),
]
df_baru = spark.createDataFrame(
    skenario, ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]
)

df_baru_idx = model_idx_menu.transform(df_baru)
df_baru_idx = model_idx_cabang.transform(df_baru_idx)
df_baru_idx = model_idx_waktu.transform(df_baru_idx)

df_baru_final = assembler.transform(df_baru_idx)
hasil = model.transform(df_baru_final)

hasil.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan",
    (col("qty") * col("harga_satuan")).alias("qty_x_harga"),
    spark_round("prediction", 0).alias("prediksi_total_penjualan")
).show(truncate=False)

spark.stop()

+----------+----------+-----------+---+------------+-----------+------------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|qty_x_harga|prediksi_total_penjualan|
+----------+----------+-----------+---+------------+-----------+------------------------+
|Kopi Susu |Magelang  |Pagi       |2  |18000       |36000      |31480.0                 |
|Roti Bakar|Yogyakarta|Siang      |3  |25000       |75000      |75276.0                 |
|Es Teh    |Semarang  |Malam      |5  |15000       |75000      |89203.0                 |
+----------+----------+-----------+---+------------+-----------+------------------------+

